In [1]:
import pandas as pd
from ete3 import Tree
import os
pd.set_option('display.max_rows', 100)  # Replace 100 with the desired number of rows

In [ ]:
from pathlib import Path

# Set up project paths
project_root = Path('/workspaces/CellTreeBench')
data_dir = project_root / 'data' / 'celegans_small' / 'P0' / 'tree_building' / 'tree_df'
out_dir = project_root / 'data' / 'celegans_small' / 'P0' / 'tree_building' / 'tree_df_adjusted'

# Ensure output directory exists
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Input directory: {data_dir}")
print(f"Output directory: {out_dir}")
# Def a function to drop nodes from tree_df
# Given the tree_df and a list of node names to drop
# Remove rows if row['Lineage'] or row['Parent'] is in the list of nodes to drop
def drop_nodes(tree_df, nodes_to_drop):
    # Drop rows where 'Lineage' is in nodes_to_drop
    tree_df = tree_df[~tree_df['Lineage'].isin(nodes_to_drop)]
    # Drop rows where 'Parent' is in nodes_to_drop
    tree_df = tree_df[~tree_df['Parent'].isin(nodes_to_drop)]
    return tree_df

def create_tree(tree_df):
    # Create the root of the tree
    node_dict = {}
    root_name = tree_df["Lineage"].iloc[0]
    tree = Tree(name=root_name)
    tree.add_feature("n_cells", tree_df["n_cells"].iloc[0])

    node_dict[root_name] = tree

    # Create nodes and arrange by parent
    for index, row in tree_df.iterrows():
        lineage_name = row["Lineage"]
        parent_name = row["Parent"]
        # Skip nodes with n_cells = 0
        # if row["n_cells"] == 0 or lineage_name == root_name:
        if lineage_name == root_name:
            continue

        # Add current node if it doesn't exist
        if lineage_name not in node_dict:
            node = Tree(name=lineage_name)
            node.add_feature("n_cells", row["n_cells"])
            node_dict[lineage_name] = node

        # Ensure parent node exists
        if parent_name not in node_dict:
            # node_dict[parent_name] = Tree(name=parent_name)
            print(f"ERR: Parent node {parent_name} not found for {lineage_name}.")

        # Attach the current node to its parent
        node_dict[parent_name].add_child(node_dict[lineage_name])
    return tree

In [ ]:
# Define lineages and their corresponding nodes to drop
lineage_adjustments = {
    "MSx": ['MSaaapaa', "MSaaaaa", "MSaaaap", "MSpappa", "MSaaaaapa"],
    "Exx": ['Epxaa', 'Epxpa'],
    "Cx": ['Caapa', 'Caappv', 'Cxpxap'],
    "Dx": ['Dxapx', 'Dxapp'],
}

def process_lineage_adjustment(lineage_name, drop_list):
    """Process a single lineage: load, adjust, save tree and files"""
    print(f"\n{'='*40}")
    print(f"Processing: {lineage_name}")
    print(f"{'='*40}")
    
    # Load tree dataframe
    file_name = data_dir / f"tree_df-{lineage_name}.csv"
    if not file_name.exists():
        print(f"Warning: File not found: {file_name}")
        return False
        
    the_df = pd.read_csv(file_name)
    print(f"Loaded tree with {len(the_df)} nodes")
    
    # Drop problematic nodes
    if len(drop_list) > 0:
        print(f"Dropping nodes: {drop_list}")
        the_df = drop_nodes(the_df, nodes_to_drop=drop_list)
        print(f"After dropping: {len(the_df)} nodes remain")
    
    # Create tree
    the_tree = create_tree(the_df)
    n_leaves = len(the_tree.get_leaf_names())
    print(f"Tree created with {n_leaves} leaves")
    
    # Save ASCII representation
    file_name = out_dir / f"{lineage_name}.txt"
    with open(file_name, "w") as f:
        f.write(the_tree.get_ascii(attributes=["name"]))
    print(f"Saved ASCII to: {file_name}")
    
    # Save ASCII with cell counts
    file_name = out_dir / f"{lineage_name}-ncells.txt"
    with open(file_name, "w") as f:
        f.write(the_tree.get_ascii(attributes=["name", "n_cells"]))
    print(f"Saved ASCII with cell counts to: {file_name}")
    
    # Save modified tree dataframe
    file_name = out_dir / f"tree_df-{lineage_name}.csv"
    the_df.to_csv(file_name, index=False)
    print(f"Saved adjusted tree dataframe to: {file_name}")
    
    return True

# Process all lineages
results = {}
for lineage_name, drop_list in lineage_adjustments.items():
    success = process_lineage_adjustment(lineage_name, drop_list)
    results[lineage_name] = success

print(f"\n{'='*60}")
print("ADJUSTMENT COMPLETE!")
print(f"{'='*60}")
print("Summary of processed lineages:")
for lineage, success in results.items():
    status = "✅ Success" if success else "❌ Failed"
    print(f"  {lineage}: {status}")
print(f"\nFiles saved to: {out_dir}")

In [9]:
lineage_name = "ABpxp"
file_name = os.path.join(data_dir, f"tree_df-{lineage_name}.csv")
the_df = pd.read_csv(file_name)

# Drop 
# Be careful with "ABpxpapa"
drop_list = ["ABprppppapa", "ABprppppapp", "ABpxpapa", "ABpxpapaa", "ABpxpapaaa", "ABpxpapap", "ABpxpapapa"]
if len(drop_list) > 0:
    the_df = drop_nodes(the_df, nodes_to_drop=drop_list)

# Add the new rows to the DataFrame
new_rows = [
    {'Lineage': 'ABprppppap', 'Parent': 'ABpxppppap', 'Level': '', 'n_cells': 0},
    {'Lineage': 'ABprppppapa', 'Parent': 'ABprppppap', 'Level': '', 'n_cells': 35},
    {'Lineage': 'ABprppppapp', 'Parent': 'ABprppppap', 'Level': '', 'n_cells': 48}
]
if len(new_rows) > 0:
    the_df = pd.concat([the_df, pd.DataFrame(new_rows)], ignore_index=True)

# Create the tree
the_tree = create_tree(the_df)
# save the get_ascii to a file
file_name = os.path.join(out_dir, f"{lineage_name}.txt")
with open(file_name, "w") as f:
    f.write(the_tree.get_ascii(attributes=["name"]))
file_name = os.path.join(out_dir, f"{lineage_name}-ncells.txt")
with open(file_name, "w") as f:
    f.write(the_tree.get_ascii(attributes=["name", "n_cells"]))
# Save the modified tree_df
file_name = os.path.join(out_dir, f"tree_df-{lineage_name}.csv")
the_df.to_csv(file_name, index=False)

In [10]:
lineage_name = "ABpxax"
file_name = os.path.join(data_dir, f"tree_df-{lineage_name}.csv")
the_df = pd.read_csv(file_name)

# Drop 
drop_list = ["ABpxapp", "ABpxappaa", "ABpxappap", "ABpxapppa", "ABpxapppp", "ABpxapppaaa", "ABpxapppapp"]
if len(drop_list) > 0:
    the_df = drop_nodes(the_df, nodes_to_drop=drop_list)

# Add the new rows to the DataFrame
new_rows = [
]
if len(new_rows) > 0:
    the_df = pd.concat([the_df, pd.DataFrame(new_rows)], ignore_index=True)

# Create the tree
the_tree = create_tree(the_df)
# save the get_ascii to a file
file_name = os.path.join(out_dir, f"{lineage_name}.txt")
with open(file_name, "w") as f:
    f.write(the_tree.get_ascii(attributes=["name"]))
file_name = os.path.join(out_dir, f"{lineage_name}-ncells.txt")
with open(file_name, "w") as f:
    f.write(the_tree.get_ascii(attributes=["name", "n_cells"]))
# Save the modified tree_df
file_name = os.path.join(out_dir, f"tree_df-{lineage_name}.csv")
the_df.to_csv(file_name, index=False)

In [11]:
lineage_name = "ABaxx"
file_name = os.path.join(data_dir, f"tree_df-{lineage_name}.csv")
the_df = pd.read_csv(file_name)

# Drop 
drop_list = ["ABarpax"]
if len(drop_list) > 0:
    the_df = drop_nodes(the_df, nodes_to_drop=drop_list)

# Add the new rows to the DataFrame
new_rows = [
]
if len(new_rows) > 0:
    the_df = pd.concat([the_df, pd.DataFrame(new_rows)], ignore_index=True)

# Create the tree
the_tree = create_tree(the_df)
# save the get_ascii to a file
file_name = os.path.join(out_dir, f"{lineage_name}.txt")
with open(file_name, "w") as f:
    f.write(the_tree.get_ascii(attributes=["name"]))
file_name = os.path.join(out_dir, f"{lineage_name}-ncells.txt")
with open(file_name, "w") as f:
    f.write(the_tree.get_ascii(attributes=["name", "n_cells"]))
# Save the modified tree_df
file_name = os.path.join(out_dir, f"tree_df-{lineage_name}.csv")
the_df.to_csv(file_name, index=False)